# Breakout behavioural analysis

## Overview

This notebook reproduces the behavioural analysis used in the thesis for **Breakout** and is intended to be included directly in the project repository.

### Contents
- Load and validate gameplay logs
- Compute episode-level behavioural metrics
- Compare human players with PPO and DQN agents
- Generate summary figures and statistical analyses

### Expected project structure

```text
project_root/
├── data/
│   ├── human/
│   ├── ppo/
│   └── dqn/
└── notebooks/
```

The data-loading section automatically discovers the required CSV files from the configured data directory. All datasets are loaded through the same helper functions to keep preprocessing consistent across human and agent data.

The notebook is written so that every section can be executed sequentially from top to bottom.


In [ ]:
# Install the required packages once in your environment if needed.
# This cell is intentionally left empty for repository use.

In [ ]:
# ============================================================
# 0. Imports and configuration
# ============================================================

from pathlib import Path
from collections import Counter
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

warnings.filterwarnings("ignore", category=FutureWarning)

# Change this if your CSVs are stored somewhere else.
DATA_DIR = Path(".")
OUT_DIR = DATA_DIR / "breakout_full_analysis_outputs"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

GAME = "Breakout"
ACTION_LABELS = {0: "NOOP", 1: "FIRE", 2: "RIGHT", 3: "LEFT"}

# In Breakout, FIRE is mainly tied to serving/start handling and is partly automated in the
# collection/evaluation setup. For action-distribution and motif reporting, the notebook
# therefore focuses on movement/control actions only. The raw action column is kept intact.
FIRE_ACTION = 1
REPORT_ACTION_LABELS = {0: "NOOP", 2: "RIGHT", 3: "LEFT"}
REPORT_ACTIONS = list(REPORT_ACTION_LABELS.keys())

SOURCE_ORDER = ["Human", "PPO", "DQN"]

# Human logs are raw ALE-frame rows. Agent logs are 4-frame decision rows.
AGENT_RAW_FRAMES_PER_ROW = 4
HUMAN_RAW_FRAMES_PER_ROW = 1

# Breakout/Pong were capped at 10,000 logged timesteps.
# Some wrappers stop at e.g. 9994/9996 rather than exactly 10000, so use a small tolerance.
CAP_THRESHOLD_LOGGED_ROWS = 9990

BASE_COLS = [
    "run_ts", "episode", "step", "action", "reward", "done", "episode_return",
    "lives_pre", "lives_post"
]
RAM_COLS = [f"ram_pre_{i}" for i in range(128)]
REQUIRED_COLS = BASE_COLS + RAM_COLS

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUT_DIR.resolve())

## 1. Locate the input data

All CSV files are discovered automatically from the configured data directory.

Using a single discovery routine keeps the loading process identical across notebooks and avoids hard-coded filenames. Human, PPO and DQN datasets are identified from their filenames before being validated and loaded.


In [ ]:
# ============================================================
# 1. Locate input files
# ============================================================

all_csvs = sorted(DATA_DIR.glob("*.csv"))

# Discover available CSV files
dqn_candidates = sorted(DATA_DIR.glob("logs/dqn_agent_breakout_30ep.csv"), key=lambda p: ("(1)" in p.name, p.name))
ppo_candidates = sorted(DATA_DIR.glob("logs/ppo_agent_breakout_30ep.csv"), key=lambda p: ("(1)" in p.name, p.name))
human_files = sorted(DATA_DIR.glob("logs/human/*.csv"))

if not dqn_candidates:
    raise FileNotFoundError("No DQN Breakout file found. Expected: dqn_agent_breakout_30ep.csv")
if not ppo_candidates:
    raise FileNotFoundError("No PPO Breakout file found. Expected: ppo_agent_breakout_30ep.csv")
if not human_files:
    raise FileNotFoundError("No human Breakout files found. Expected files matching: human_pixels_*.csv")

DQN_FILE = dqn_candidates[0]
PPO_FILE = ppo_candidates[0]

print("Using DQN file:", DQN_FILE.name)
print("Using PPO file:", PPO_FILE.name)
print("Number of human logs found:", len(human_files))
print("First few human files:")
for f in human_files[:5]:
    print(" -", f.name)

## 2. Load and validate the logs

For agents, one row is one selected action repeated for 4 raw ALE frames. For humans, one row is one raw ALE frame. The notebook keeps this distinction explicit.

In [ ]:
# ============================================================
# 2. Loading helpers
# ============================================================

def validate_columns(path):
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in REQUIRED_COLS if c not in cols]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing[:10]}{'...' if len(missing) > 10 else ''}")
    return True

for path in [DQN_FILE, PPO_FILE] + human_files:
    validate_columns(path)


def load_agent(path, source):
    df = pd.read_csv(path, usecols=REQUIRED_COLS)
    df["source"] = source
    df["game"] = GAME
    df["orig_file"] = path.name
    df["global_episode"] = df["episode"].astype(int)
    df["collection_scale"] = "agent_decision_step"
    df["raw_frames_per_row"] = AGENT_RAW_FRAMES_PER_ROW
    return df


def load_human_files(files):
    parts = []
    episode_counter = 0
    for f in files:
        df = pd.read_csv(f, usecols=REQUIRED_COLS)
        # Usually each file is one episode. This also handles the case where a file contains multiple episodes.
        for local_episode, sub in df.groupby("episode", sort=False):
            episode_counter += 1
            sub = sub.copy()
            sub["source"] = "Human"
            sub["game"] = GAME
            sub["orig_file"] = f.name
            sub["global_episode"] = episode_counter
            sub["collection_scale"] = "human_raw_frame"
            sub["raw_frames_per_row"] = HUMAN_RAW_FRAMES_PER_ROW
            parts.append(sub)
    return pd.concat(parts, ignore_index=True)


dqn_raw = load_agent(DQN_FILE, "DQN")
ppo_raw = load_agent(PPO_FILE, "PPO")
human_raw = load_human_files(human_files)

raw_all = pd.concat([human_raw, ppo_raw, dqn_raw], ignore_index=True)

validation_summary = (
    raw_all.groupby("source")
    .agg(
        episodes=("global_episode", "nunique"),
        rows=("action", "size"),
        files=("orig_file", "nunique"),
        min_action=("action", "min"),
        max_action=("action", "max")
    )
    .reindex(SOURCE_ORDER)
)

validation_summary.to_csv(TABLE_DIR / "01_validation_summary.csv")
validation_summary

## 3. Episode-level performance descriptors

Capped episodes are kept as valid traces. They are marked as `truncated_at_cap`, not removed.

In [ ]:
# ============================================================
# 3. Episode-level performance metrics
# ============================================================

def episode_summary(df):
    rows = []
    for (source, ep), g in df.groupby(["source", "global_episode"], sort=True):
        g = g.sort_values("step")
        raw_mult = int(g["raw_frames_per_row"].iloc[0])
        logged_rows = len(g)
        raw_frames = logged_rows * raw_mult

        rewards = g["reward"].to_numpy()
        nonzero_positions = np.flatnonzero(rewards != 0)
        if len(nonzero_positions) > 0:
            first_pos = int(nonzero_positions[0])
            first_reward_latency_raw = (first_pos + 1) * raw_mult
        else:
            first_reward_latency_raw = np.nan

        if {"lives_pre", "lives_post"}.issubset(g.columns):
            life_losses = int((g["lives_post"].to_numpy() < g["lives_pre"].to_numpy()).sum())
        else:
            life_losses = np.nan

        # Cap is based on logged rows, because TimeLimit was applied at logged environment-step scale.
        capped = logged_rows >= CAP_THRESHOLD_LOGGED_ROWS
        end_type = "truncated_at_cap" if capped else "natural_terminal"

        final_return = float(g["episode_return"].iloc[-1])
        reward_sum = float(g["reward"].sum())

        rows.append({
            "game": GAME,
            "source": source,
            "episode": ep,
            "orig_file": g["orig_file"].iloc[0],
            "logged_rows": logged_rows,
            "raw_frames_per_row": raw_mult,
            "raw_frames": raw_frames,
            "final_return": final_return,
            "reward_sum": reward_sum,
            "return_minus_reward_sum": final_return - reward_sum,
            "reward_density_per_1000_raw": final_return / raw_frames * 1000 if raw_frames > 0 else np.nan,
            "first_reward_latency_raw": first_reward_latency_raw,
            "life_losses": life_losses,
            "done_last": int(g["done"].iloc[-1]),
            "end_type": end_type,
        })
    return pd.DataFrame(rows)


episode_metrics = episode_summary(raw_all)
episode_metrics.to_csv(TABLE_DIR / "02_episode_level_metrics.csv", index=False)

# Useful display table.
episode_counts = pd.crosstab(episode_metrics["source"], episode_metrics["end_type"]).reindex(SOURCE_ORDER).fillna(0).astype(int)
episode_counts.to_csv(TABLE_DIR / "03_episode_end_type_counts.csv")
episode_counts

In [ ]:
def summarize_numeric(df, group_col="source", metrics=None):
    if metrics is None:
        metrics = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    rows = []
    for group, g in df.groupby(group_col):
        row = {group_col: group, "n": len(g)}
        for m in metrics:
            x = g[m].dropna()
            row[f"{m}_mean"] = x.mean()
            row[f"{m}_sd"] = x.std(ddof=1)
            row[f"{m}_median"] = x.median()
            row[f"{m}_iqr"] = x.quantile(0.75) - x.quantile(0.25)
            row[f"{m}_min"] = x.min()
            row[f"{m}_max"] = x.max()
        rows.append(row)
    return pd.DataFrame(rows).set_index(group_col).reindex(SOURCE_ORDER).reset_index()

performance_metrics = [
    "final_return", "raw_frames", "reward_density_per_1000_raw",
    "first_reward_latency_raw", "life_losses"
]

performance_summary = summarize_numeric(episode_metrics, metrics=performance_metrics)
performance_summary.to_csv(TABLE_DIR / "04_performance_summary.csv", index=False)
performance_summary

## 4. Align human traces to 4-frame decision windows

Human gameplay was collected responsively at raw-frame scale. For action and RAM descriptors, human traces are aggregated into non-overlapping 4-frame windows:

- action = modal action in the window,
- ties = earliest action in the window,
- reward = sum of rewards in the window,
- RAM state = first `ram_pre` vector in the window.

This preserves the thesis distinction between responsive human collection and comparable analysis units.

In [ ]:
# ============================================================
# 4. Temporal alignment
# ============================================================

def modal_action_earliest_tie(actions):
    actions = list(map(int, actions))
    counts = Counter(actions)
    max_count = max(counts.values())
    tied = {a for a, c in counts.items() if c == max_count}
    return next(a for a in actions if a in tied)


def aggregate_human_to_4frame_windows(df):
    parts = []
    for ep, g in df.groupby("global_episode", sort=True):
        g = g.sort_values("step").reset_index(drop=True)
        g["analysis_window"] = np.arange(len(g)) // 4
        ram_arr = g[RAM_COLS].to_numpy(dtype=np.uint8)

        rows = []
        ram_rows = []
        for w, idx in g.groupby("analysis_window").indices.items():
            idx = list(idx)
            sub = g.iloc[idx]
            rows.append({
                "game": GAME,
                "source": "Human",
                "global_episode": ep,
                "step": int(w),
                "action": modal_action_earliest_tie(sub["action"].tolist()),
                "reward": float(sub["reward"].sum()),
                "done": int(sub["done"].max()),
                "episode_return": float(sub["episode_return"].iloc[-1]),
                "lives_pre": int(sub["lives_pre"].iloc[0]),
                "lives_post": int(sub["lives_post"].iloc[-1]),
                "orig_file": sub["orig_file"].iloc[0],
                "collection_scale": "aligned_4frame_window",
                "raw_frames_per_row": 4,
            })
            ram_rows.append(ram_arr[idx[0]])

        meta = pd.DataFrame(rows)
        ram_df = pd.DataFrame(np.vstack(ram_rows), columns=RAM_COLS)
        parts.append(pd.concat([meta.reset_index(drop=True), ram_df.reset_index(drop=True)], axis=1))

    return pd.concat(parts, ignore_index=True)


human_aligned = aggregate_human_to_4frame_windows(human_raw)

agent_aligned = pd.concat([ppo_raw, dqn_raw], ignore_index=True).copy()
agent_aligned["collection_scale"] = "aligned_4frame_window"
agent_aligned["raw_frames_per_row"] = 4

aligned = pd.concat([
    human_aligned[["game", "source", "global_episode", "step", "action", "reward", "done", "episode_return", "lives_pre", "lives_post", "orig_file", "collection_scale", "raw_frames_per_row"] + RAM_COLS],
    agent_aligned[["game", "source", "global_episode", "step", "action", "reward", "done", "episode_return", "lives_pre", "lives_post", "orig_file", "collection_scale", "raw_frames_per_row"] + RAM_COLS]
], ignore_index=True)

alignment_summary = (
    aligned.groupby("source")
    .agg(episodes=("global_episode", "nunique"), aligned_decision_windows=("action", "size"))
    .reindex(SOURCE_ORDER)
)
alignment_summary.to_csv(TABLE_DIR / "05_alignment_summary.csv")
alignment_summary

## 5. Action-structure descriptors

These are computed on aligned 4-frame decision windows for all sources.

For Breakout, `FIRE` is excluded from the main action-structure analysis because it is mainly tied to serve/start handling and is partly automated in the setup. This means that action distributions, entropy, switching rate, run length, trigram diversity, four-gram diversity, and motif tables are all computed on the remaining paddle-control sequence only: `NOOP`, `RIGHT`, and `LEFT`.

A separate FIRE-removal table is exported so it remains transparent how many aligned windows were excluded per source.


In [ ]:
# ============================================================
# 5. Action-structure metrics
# ============================================================

# Note on FIRE in Breakout:
# FIRE is mainly related to serving/start handling and is partly automated by the evaluation setup.
# To avoid over-interpreting FIRE as a meaningful movement/control action, the main
# action-structure descriptors below are computed after removing FIRE windows.
# The resulting action sequence focuses on paddle-control behaviour: NOOP, RIGHT, LEFT.

def filter_fire(actions):
    """Remove FIRE actions and return a numpy array of NOOP/RIGHT/LEFT actions."""
    actions = np.asarray(actions, dtype=int)
    return actions[actions != FIRE_ACTION]


def action_entropy_bits(actions):
    """Entropy over the observed non-FIRE action labels only.

    Because FIRE is excluded, the maximum possible entropy is log2(3) ≈ 1.585 bits
    for Breakout's NOOP/RIGHT/LEFT control actions.
    """
    actions = np.asarray(actions, dtype=int)
    if len(actions) == 0:
        return np.nan
    counts = pd.Series(actions).value_counts().reindex(REPORT_ACTIONS, fill_value=0).to_numpy()
    probs = counts[counts > 0] / counts.sum()
    return float(-(probs * np.log2(probs)).sum()) if counts.sum() > 0 else np.nan


def switching_rate(actions):
    actions = np.asarray(actions, dtype=int)
    if len(actions) < 2:
        return np.nan
    return float(np.mean(actions[1:] != actions[:-1]))


def mean_run_length(actions):
    actions = list(map(int, actions))
    if not actions:
        return np.nan
    runs = []
    current = actions[0]
    length = 1
    for a in actions[1:]:
        if a == current:
            length += 1
        else:
            runs.append(length)
            current = a
            length = 1
    runs.append(length)
    return float(np.mean(runs))


def ngram_diversity(actions, n):
    actions = list(map(int, actions))
    if len(actions) < n:
        return np.nan
    grams = [tuple(actions[i:i+n]) for i in range(len(actions)-n+1)]
    return len(set(grams)) / len(grams)


def top_ngrams(actions, n, k=10):
    actions = list(map(int, actions))
    if len(actions) < n:
        return []
    grams = [tuple(actions[i:i+n]) for i in range(len(actions)-n+1)]
    return Counter(grams).most_common(k)


def non_fire_proportions(actions_non_fire):
    """Return proportions for NOOP/RIGHT/LEFT after excluding FIRE windows."""
    actions_non_fire = np.asarray(actions_non_fire, dtype=int)
    denom = len(actions_non_fire)
    out = {}
    for action_id, label in REPORT_ACTION_LABELS.items():
        out[f"prop_{label}"] = float(np.mean(actions_non_fire == action_id)) if denom > 0 else np.nan
    return out


# Transparency table: how much FIRE was removed before action-structure analysis.
fire_exclusion_rows = []
for source, g in aligned.groupby("source"):
    total = len(g)
    n_fire = int((g["action"].astype(int) == FIRE_ACTION).sum())
    fire_exclusion_rows.append({
        "source": source,
        "total_aligned_windows": total,
        "fire_windows_excluded": n_fire,
        "non_fire_windows_retained": total - n_fire,
        "fire_share_excluded": n_fire / total if total > 0 else np.nan,
    })

fire_exclusion_table = (
    pd.DataFrame(fire_exclusion_rows)
    .set_index("source")
    .reindex(SOURCE_ORDER)
    .reset_index()
)
fire_exclusion_table.to_csv(TABLE_DIR / "06_fire_windows_excluded_before_action_analysis.csv", index=False)
display(fire_exclusion_table)


action_rows = []
for (source, ep), g in aligned.groupby(["source", "global_episode"], sort=True):
    g = g.sort_values("step")
    actions_full = g["action"].astype(int).to_numpy()
    actions = filter_fire(actions_full)

    row = {
        "source": source,
        "episode": ep,
        "n_decision_windows_full": len(actions_full),
        "n_fire_windows_excluded": int(np.sum(actions_full == FIRE_ACTION)),
        "n_non_fire_windows": len(actions),
        "fire_share_excluded": float(np.mean(actions_full == FIRE_ACTION)) if len(actions_full) > 0 else np.nan,
        "action_entropy_bits": action_entropy_bits(actions),
        "switching_rate": switching_rate(actions),
        "mean_run_length": mean_run_length(actions),
        "trigram_diversity": ngram_diversity(actions, 3),
        "fourgram_diversity": ngram_diversity(actions, 4),
    }
    row.update(non_fire_proportions(actions))
    action_rows.append(row)

action_episode_metrics = pd.DataFrame(action_rows)

# All main action-structure columns now exclude FIRE consistently.
action_episode_metrics.to_csv(TABLE_DIR / "07_action_episode_metrics_no_fire.csv", index=False)

action_summary = summarize_numeric(
    action_episode_metrics,
    metrics=[
        "n_decision_windows_full", "n_fire_windows_excluded", "n_non_fire_windows", "fire_share_excluded",
        "action_entropy_bits", "switching_rate", "mean_run_length", "trigram_diversity", "fourgram_diversity"
    ] + [f"prop_{x}" for x in REPORT_ACTION_LABELS.values()]
)
action_summary.to_csv(TABLE_DIR / "08_action_summary_no_fire.csv", index=False)
action_summary


## 5b. Aggregation-rule robustness check

The main action-structure descriptors aggregate each non-overlapping 4-frame human window to its **modal action (earliest on tie)**. Because NOOP is the plurality action at raw-frame resolution, this rule can mechanically inflate the human NOOP share, lower switching/entropy, and lengthen runs. This cell re-aggregates the **human** traces under alternative within-window rules — `last` (last action in the window) and `modal_movement` (modal, but ties resolve toward RIGHT/LEFT rather than NOOP) — and recomputes entropy, switching rate, and mean run length. Agents are unchanged (already at the decision scale), and FIRE handling is applied **after** aggregation exactly as in the main pipeline. The verdict reports whether the Human-vs-agent ordering survives every rule. A `raw_frame` row is included as a no-aggregation reference to show how far the aggregation step itself moves the human numbers. The `modal_earliest` row reproduces the main table and serves as a correctness check.

In [ ]:
# ============================================================
# 5b. Aggregation-rule robustness check (action structure)
# ============================================================
# Tests whether the Human-vs-agent ordering on entropy, switching rate, and
# mean run length survives changing the within-window aggregation rule.
# Only the HUMAN traces are re-aggregated; agents are already at the 4-frame
# decision scale and are unchanged. FIRE handling is applied AFTER aggregation,
# exactly as in the main pipeline. Reuses the descriptor functions and the
# `aligned` / `human_raw` dataframes already defined above.

from collections import Counter

MOVEMENT_ACTIONS = {2, 3}            # RIGHT, LEFT (NOOP=0, FIRE=1 are non-movement)
RULES = ["modal_earliest", "last", "modal_movement"]

def representative_action(window_actions, rule):
    wa = list(map(int, window_actions))
    if rule == "last":
        return wa[-1]
    counts = Counter(wa)
    mx = max(counts.values())
    tied = [a for a in wa if counts[a] == mx]   # window order; tied[0] == earliest-tie
    if rule == "modal_earliest":
        return tied[0]
    if rule == "modal_movement":                # prefer a movement action on ties
        for a in tied:
            if a in MOVEMENT_ACTIONS:
                return a
        return tied[0]
    raise ValueError(rule)

def _episode_descriptors(action_seq):
    a = filter_fire(action_seq)                 # exclude FIRE AFTER aggregation
    return (action_entropy_bits(a), switching_rate(a), mean_run_length(a))

def human_means_under_rule(rule):
    e, s, r = [], [], []
    for ep, g in human_raw.groupby("global_episode", sort=True):
        acts = g.sort_values("step")["action"].astype(int).to_numpy()
        win = np.arange(len(acts)) // 4
        reps = np.array([representative_action(acts[win == w], rule) for w in np.unique(win)], dtype=int)
        ent, sw, rl = _episode_descriptors(reps)
        e.append(ent); s.append(sw); r.append(rl)
    return np.nanmean(e), np.nanmean(s), np.nanmean(r)

def human_means_raw_frame():
    e, s, r = [], [], []
    for ep, g in human_raw.groupby("global_episode", sort=True):
        acts = g.sort_values("step")["action"].astype(int).to_numpy()
        ent, sw, rl = _episode_descriptors(acts)
        e.append(ent); s.append(sw); r.append(rl)
    return np.nanmean(e), np.nanmean(s), np.nanmean(r)

def agent_means(source):
    e, s, r = [], [], []
    for ep, g in aligned[aligned["source"] == source].groupby("global_episode", sort=True):
        acts = g.sort_values("step")["action"].astype(int).to_numpy()
        ent, sw, rl = _episode_descriptors(acts)
        e.append(ent); s.append(sw); r.append(rl)
    return np.nanmean(e), np.nanmean(s), np.nanmean(r)

ppo_vals = dict(zip(["entropy", "switching", "run_length"], agent_means("PPO")))
dqn_vals = dict(zip(["entropy", "switching", "run_length"], agent_means("DQN")))

# Build the long-form robustness table.
robust_rows = []
human_by_rule = {}
for rule in RULES:
    hv = dict(zip(["entropy", "switching", "run_length"], human_means_under_rule(rule)))
    human_by_rule[rule] = hv
    for metric in ["entropy", "switching", "run_length"]:
        robust_rows.append({
            "metric": metric, "aggregation_rule": rule,
            "Human": round(hv[metric], 3),
            "PPO": round(ppo_vals[metric], 3),
            "DQN": round(dqn_vals[metric], 3),
        })
robustness_table = pd.DataFrame(robust_rows)

# Verdict: does Human stay the extreme value under EVERY aggregation rule?
#   entropy & switching -> Human should be the lowest; run_length -> Human highest.
def verdict(metric):
    extreme_is_min = metric in ("entropy", "switching")
    for rule in RULES:
        h = human_by_rule[rule][metric]; p = ppo_vals[metric]; d = dqn_vals[metric]
        ok = (h < min(p, d)) if extreme_is_min else (h > max(p, d))
        if not ok:
            return f"SENSITIVE (fails under '{rule}')"
    return "ROBUST (holds under all rules)"

verdict_table = pd.DataFrame([{"metric": m, "verdict": verdict(m)}
                              for m in ["entropy", "switching", "run_length"]])

# Aggregation-sensitivity diagnostic: how far the raw-frame human sits from each rule.
raw_h = dict(zip(["entropy", "switching", "run_length"], human_means_raw_frame()))
sens_rows = [{"aggregation": "raw_frame (1-frame, reference)", **{k: round(raw_h[k], 3) for k in raw_h}}]
for rule in RULES:
    sens_rows.append({"aggregation": rule, **{k: round(human_by_rule[rule][k], 3) for k in human_by_rule[rule]}})
sensitivity_table = pd.DataFrame(sens_rows)

robustness_table.to_csv(TABLE_DIR / "26_aggregation_robustness_action_structure.csv", index=False)
verdict_table.to_csv(TABLE_DIR / "27_aggregation_robustness_verdict.csv", index=False)
sensitivity_table.to_csv(TABLE_DIR / "28_aggregation_sensitivity_human_raw_vs_window.csv", index=False)

# Baseline reproduction check: modal_earliest here should match the main table (cell with compact_action).
print("Baseline (modal_earliest) Human reproduction check:")
print("  entropy={:.3f}  switching={:.3f}  run_length={:.3f}".format(
    human_by_rule["modal_earliest"]["entropy"],
    human_by_rule["modal_earliest"]["switching"],
    human_by_rule["modal_earliest"]["run_length"]))
print("  (compare to compact_action / table 23 Human row)\n")
display(robustness_table)
display(verdict_table)
display(sensitivity_table)


In [ ]:
# Time-weighted action distribution: every aligned non-FIRE decision window counts.
# FIRE is excluded and proportions are renormalized over NOOP/RIGHT/LEFT.
action_distribution_source = aligned[aligned["action"].astype(int) != FIRE_ACTION].copy()
action_distribution_time = (
    action_distribution_source.groupby(["source", "action"])
    .size()
    .rename("count")
    .reset_index()
)
action_distribution_time["action_label"] = action_distribution_time["action"].map(REPORT_ACTION_LABELS)
action_distribution_time["proportion"] = action_distribution_time.groupby("source")["count"].transform(lambda x: x / x.sum())
action_distribution_time.to_csv(TABLE_DIR / "09_action_distribution_time_weighted_no_fire.csv", index=False)

# Episode-averaged action distribution: every episode contributes equally.
# Uses the same FIRE-excluded, renormalized proportions.
episode_avg_rows = []
for source, g in action_episode_metrics.groupby("source"):
    row = {"source": source}
    for label in REPORT_ACTION_LABELS.values():
        row[label] = g[f"prop_{label}"].mean()
    episode_avg_rows.append(row)
action_distribution_episode_avg = pd.DataFrame(episode_avg_rows).set_index("source").reindex(SOURCE_ORDER).reset_index()
action_distribution_episode_avg.to_csv(TABLE_DIR / "10_action_distribution_episode_averaged_no_fire.csv", index=False)

action_distribution_time.pivot(index="action_label", columns="source", values="proportion").reindex(REPORT_ACTION_LABELS.values())


In [ ]:
# Motif tables, time-weighted across the full aligned sequence per source.
# FIRE is excluded before motif extraction, so motifs focus on NOOP/RIGHT/LEFT control patterns.
motif_rows = []
for source, g in aligned.groupby("source"):
    actions_full = g.sort_values(["global_episode", "step"])["action"].astype(int).to_numpy()
    actions = filter_fire(actions_full)
    for n in [3, 4]:
        denom = max(1, len(actions) - n + 1)
        for motif, count in top_ngrams(actions, n=n, k=15):
            motif_rows.append({
                "source": source,
                "n": n,
                "motif": "-".join(REPORT_ACTION_LABELS[i] for i in motif),
                "count": count,
                "proportion": count / denom,
            })

motif_table = pd.DataFrame(motif_rows)
motif_table.to_csv(TABLE_DIR / "11_top_motifs_time_weighted_no_fire.csv", index=False)
motif_table.head(20)


## 6. Descriptive effect sizes and distribution distances

The thesis is descriptive, so the notebook reports effect sizes rather than relying on hypothesis tests.

In [ ]:
# ============================================================
# 6. Effect sizes and distribution distances
# ============================================================

PAIRS = [("Human", "PPO"), ("Human", "DQN"), ("PPO", "DQN")]

def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna())
    y = np.asarray(pd.Series(y).dropna())
    if len(x) == 0 or len(y) == 0:
        return np.nan
    greater = sum((xi > y).sum() for xi in x)
    lower = sum((xi < y).sum() for xi in x)
    return (greater - lower) / (len(x) * len(y))


def js_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    if p.sum() == 0 or q.sum() == 0:
        return np.nan
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def total_variation_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    return 0.5 * np.abs(p - q).sum()

# Numeric episode-level/action-level effect sizes.
# Action-structure effect sizes use the FIRE-excluded paddle-control metrics from action_episode_metrics.
effect_rows = []
for metric in performance_metrics:
    for a, b in PAIRS:
        xa = episode_metrics.loc[episode_metrics["source"] == a, metric]
        xb = episode_metrics.loc[episode_metrics["source"] == b, metric]
        effect_rows.append({
            "descriptor_group": "performance",
            "metric": metric,
            "comparison": f"{a} vs {b}",
            "cliffs_delta_first_minus_second": cliffs_delta(xa, xb),
            "mean_first": xa.mean(),
            "mean_second": xb.mean(),
        })

for metric in ["action_entropy_bits", "switching_rate", "mean_run_length", "trigram_diversity", "fourgram_diversity"]:
    for a, b in PAIRS:
        xa = action_episode_metrics.loc[action_episode_metrics["source"] == a, metric]
        xb = action_episode_metrics.loc[action_episode_metrics["source"] == b, metric]
        effect_rows.append({
            "descriptor_group": "action_structure",
            "metric": metric,
            "comparison": f"{a} vs {b}",
            "cliffs_delta_first_minus_second": cliffs_delta(xa, xb),
            "mean_first": xa.mean(),
            "mean_second": xb.mean(),
        })

# Action distribution distances, time weighted. FIRE is excluded and proportions are normalized over NOOP/RIGHT/LEFT.
dist_piv = (
    action_distribution_time
    .pivot(index="source", columns="action", values="proportion")
    .reindex(index=SOURCE_ORDER, columns=REPORT_ACTIONS)
    .fillna(0)
)
for a, b in PAIRS:
    p = dist_piv.loc[a].values
    q = dist_piv.loc[b].values
    effect_rows.append({
        "descriptor_group": "action_distribution_no_fire",
        "metric": "action_distribution_JS_divergence_time_weighted_no_fire",
        "comparison": f"{a} vs {b}",
        "cliffs_delta_first_minus_second": np.nan,
        "mean_first": np.nan,
        "mean_second": np.nan,
        "js_divergence": js_divergence(p, q),
        "total_variation_distance": total_variation_distance(p, q),
    })

effect_sizes = pd.DataFrame(effect_rows)
effect_sizes.to_csv(TABLE_DIR / "12_descriptive_effect_sizes_and_distances.csv", index=False)
effect_sizes.head(20)


## 7. RAM exact state-visitation overlap

This is a strict baseline: two RAM states only match if all 128 bytes are identical.

In [ ]:
# ============================================================
# 7. Exact RAM uniqueness and overlap
# ============================================================

ram_sets = {}
unique_rows = []

for source, g in aligned.groupby("source"):
    arr = g[RAM_COLS].to_numpy(dtype=np.uint8)
    state_set = set(map(bytes, arr))
    ram_sets[source] = state_set
    unique_rows.append({
        "source_or_pair": source,
        "aligned_rows": len(arr),
        "unique_exact_ram_states": len(state_set),
        "unique_share_of_rows": len(state_set) / len(arr),
        "jaccard_overlap": np.nan,
    })

for a, b in PAIRS:
    inter = len(ram_sets[a] & ram_sets[b])
    union = len(ram_sets[a] | ram_sets[b])
    unique_rows.append({
        "source_or_pair": f"{a} ∩ {b}",
        "aligned_rows": np.nan,
        "unique_exact_ram_states": inter,
        "unique_share_of_rows": np.nan,
        "jaccard_overlap": inter / union if union else np.nan,
    })

ram_exact_overlap = pd.DataFrame(unique_rows)
ram_exact_overlap.to_csv(TABLE_DIR / "12_ram_exact_overlap.csv", index=False)
ram_exact_overlap

## 8. RAM PCA and clustering

PCA is used for visualization. Clustering gives coarse RAM-state regions. Both are fitted within Breakout only, using the aligned RAM vectors from Human, PPO, and DQN together.

**Feature refinement (revised).** Before scaling, constant and near-constant RAM bytes are dropped (bytes whose standard deviation across the pooled traces is below `NEAR_CONST_STD_THRESHOLD`, measured on the raw 0-255 scale). This addresses the limitation noted in Section 3.13: standardizing a near-constant byte divides noise-level fluctuations by a near-zero standard deviation and inflates its influence on the Euclidean distance used by PCA and k-means. The list of kept/dropped bytes and their dispersion is written to `21_ram_byte_variance_kept_dropped.csv` for transparency. Clustering then proceeds on the retained bytes only.

The default clustering uses `k = 15`, plus sensitivity checks for `k = 10` and `k = 20`. Because the clustering input has changed, the PCA figure, cluster heatmaps, cluster-difference plots, and cluster-distance tables must be re-run; exact-RAM uniqueness/overlap (Section 7), performance, and action-structure results are unaffected and do not need re-running.

In [ ]:
# ============================================================
# 8. RAM PCA and clustering
# ============================================================

# Convert RAM to numeric matrix.
# This can take some memory because DQN has many rows.
ram_matrix_full = aligned[RAM_COLS].to_numpy(dtype=np.float32)
print("RAM matrix shape (all 128 bytes):", ram_matrix_full.shape)

# ------------------------------------------------------------
# Feature refinement: drop constant / near-constant RAM bytes
# ------------------------------------------------------------
# Rationale (see thesis Section 3.13): standardizing a constant or near-constant
# byte divides tiny, possibly noise-level fluctuations by a near-zero standard
# deviation, inflating their influence on the Euclidean distance used by PCA and
# k-means. Bytes that never (or barely) vary across the pooled Human+PPO+DQN
# traces carry no information about state visitation, so they are removed before
# scaling. Bytes are measured on the raw 0-255 scale; NEAR_CONST_STD_THRESHOLD is
# the minimum standard deviation (in raw byte units) a byte must have to be kept.
NEAR_CONST_STD_THRESHOLD = 1.0

byte_std = ram_matrix_full.std(axis=0)
byte_mean = ram_matrix_full.mean(axis=0)
keep_mask = byte_std > NEAR_CONST_STD_THRESHOLD

kept_idx = np.where(keep_mask)[0]
dropped_idx = np.where(~keep_mask)[0]
kept_byte_names = [RAM_COLS[i] for i in kept_idx]

# Transparency table: which bytes were kept/dropped and their dispersion.
ram_byte_variance = pd.DataFrame({
    "ram_byte": RAM_COLS,
    "byte_index": np.arange(128),
    "mean": byte_mean,
    "std": byte_std,
    "kept": keep_mask,
}).sort_values("std", ascending=False).reset_index(drop=True)
ram_byte_variance.to_csv(TABLE_DIR / "21_ram_byte_variance_kept_dropped.csv", index=False)

print(f"Near-constant std threshold: {NEAR_CONST_STD_THRESHOLD}")
print(f"Bytes kept:    {len(kept_idx):3d} / 128")
print(f"Bytes dropped: {len(dropped_idx):3d} / 128")
print("Kept byte indices:", kept_idx.tolist())

# Use only the informative bytes for PCA and clustering.
ram_matrix = ram_matrix_full[:, keep_mask]
print("RAM matrix shape (after dropping near-constant bytes):", ram_matrix.shape)

scaler = StandardScaler()
X = scaler.fit_transform(ram_matrix).astype(np.float32)

pca = PCA(n_components=2, random_state=0)
pca_coords = pca.fit_transform(X)

# Keep metadata explicitly attached to the PCA coordinates.
# Important: do NOT use groupby.apply here, because some pandas versions drop the
# grouping column from the result. That caused the earlier missing 'source' error.
pca_df = pd.DataFrame({
    "source": aligned["source"].to_numpy(),
    "global_episode": aligned["global_episode"].to_numpy(),
    "step": aligned["step"].to_numpy(),
    "pc1": pca_coords[:, 0],
    "pc2": pca_coords[:, 1],
})

# Save a manageable, balanced sample for plotting/reporting.
# This samples up to 5,000 rows per source while preserving the source column.
PCA_SAMPLE_PER_SOURCE = 5000
sample_parts = []
for source in SOURCE_ORDER:
    sub = pca_df[pca_df["source"] == source]
    if len(sub) == 0:
        print(f"Warning: no PCA rows found for source {source}")
        continue
    n_sample = min(len(sub), PCA_SAMPLE_PER_SOURCE)
    sample_parts.append(sub.sample(n=n_sample, random_state=0))

pca_sample = pd.concat(sample_parts, ignore_index=True)

# Extra safety checks for the scatter plot.
required_pca_cols = {"source", "global_episode", "step", "pc1", "pc2"}
missing = required_pca_cols.difference(pca_sample.columns)
if missing:
    raise ValueError(f"PCA sample is missing columns: {sorted(missing)}")

pca_explained = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
})

pca_sample.to_csv(TABLE_DIR / "16_pca_sample.csv", index=False)
pca_explained.to_csv(TABLE_DIR / "17_pca_explained_variance.csv", index=False)

print("PCA sample columns:", pca_sample.columns.tolist())
print("PCA sample rows by source:")
print(pca_sample["source"].value_counts().reindex(SOURCE_ORDER))
print(pca_explained)


In [ ]:
def run_ram_clustering(X, aligned_meta, k=15, random_state=0):
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=random_state,
        batch_size=4096,
        n_init=10,
        max_iter=200,
    )
    clusters = km.fit_predict(X)
    cl = aligned_meta[["source", "global_episode", "step"]].copy()
    cl["cluster"] = clusters

    # Time-weighted: every aligned decision window counts.
    time_shares = pd.crosstab(cl["source"], cl["cluster"], normalize="index")
    time_shares = time_shares.reindex(SOURCE_ORDER)

    # Episode-averaged: first compute per-episode cluster shares, then average within source.
    ep_counts = cl.groupby(["source", "global_episode", "cluster"]).size().rename("n").reset_index()
    ep_pivot = ep_counts.pivot_table(index=["source", "global_episode"], columns="cluster", values="n", fill_value=0)
    ep_pivot = ep_pivot.div(ep_pivot.sum(axis=1), axis=0)
    ep_avg = ep_pivot.groupby(level=0).mean().reindex(SOURCE_ORDER)

    # Pairwise distances.
    dist_rows = []
    for a, b in PAIRS:
        p_time = time_shares.loc[a].values
        q_time = time_shares.loc[b].values
        p_ep = ep_avg.loc[a].values
        q_ep = ep_avg.loc[b].values
        dist_rows.append({
            "k": k,
            "comparison": f"{a} vs {b}",
            "js_divergence_time_weighted": js_divergence(p_time, q_time),
            "total_variation_time_weighted": total_variation_distance(p_time, q_time),
            "js_divergence_episode_avg": js_divergence(p_ep, q_ep),
            "total_variation_episode_avg": total_variation_distance(p_ep, q_ep),
        })

    return cl, time_shares, ep_pivot, ep_avg, pd.DataFrame(dist_rows)

cluster_meta = aligned[["source", "global_episode", "step"]].copy()

cluster_assignments_15, cluster_shares_time_15, cluster_shares_episode_15, cluster_shares_episode_avg_15, cluster_distances_15 = run_ram_clustering(X, cluster_meta, k=15)

cluster_assignments_15.to_csv(TABLE_DIR / "15_cluster_assignments_k15.csv", index=False)
cluster_shares_time_15.to_csv(TABLE_DIR / "16_cluster_shares_time_weighted_k15.csv")
cluster_shares_episode_15.to_csv(TABLE_DIR / "17_cluster_shares_by_episode_k15.csv")
cluster_shares_episode_avg_15.to_csv(TABLE_DIR / "18_cluster_shares_episode_averaged_k15.csv")
cluster_distances_15.to_csv(TABLE_DIR / "19_cluster_distances_k15.csv", index=False)

cluster_distances_15

In [ ]:
# Sensitivity checks for different cluster counts.
sensitivity_rows = []
for k in [10, 20]:
    _, time_shares, _, ep_avg, dist = run_ram_clustering(X, cluster_meta, k=k)
    time_shares.to_csv(TABLE_DIR / f"cluster_shares_time_weighted_k{k}.csv")
    ep_avg.to_csv(TABLE_DIR / f"cluster_shares_episode_averaged_k{k}.csv")
    dist.to_csv(TABLE_DIR / f"cluster_distances_k{k}.csv", index=False)
    sensitivity_rows.append(dist)

cluster_sensitivity = pd.concat([cluster_distances_15] + sensitivity_rows, ignore_index=True)
cluster_sensitivity.to_csv(TABLE_DIR / "20_cluster_distance_sensitivity_k10_k15_k20.csv", index=False)
cluster_sensitivity

## 9. Visualizations

The following figures are saved in `breakout_full_analysis_outputs/figures/`. They are also displayed in the notebook.

In [ ]:
# ============================================================
# 9. Plot helpers
# ============================================================

def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()


def source_boxplot(df, metric, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    data = [df.loc[df["source"] == s, metric].dropna().values for s in SOURCE_ORDER]
    ax.boxplot(data, tick_labels=SOURCE_ORDER, showmeans=True)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    save_show(fig, filename)


def source_bar(df, x_col, y_col, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(df[x_col]))
    ax.bar(x, df[y_col])
    ax.set_xticks(x)
    ax.set_xticklabels(df[x_col])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    save_show(fig, filename)

In [ ]:
# Performance figures.
source_boxplot(
    episode_metrics, "final_return", "Episode return",
    "Breakout episode return by source",
    "01_episode_return_boxplot.png"
)

source_boxplot(
    episode_metrics, "raw_frames", "Episode length in raw ALE frames",
    "Breakout episode length by source",
    "02_episode_length_raw_frames_boxplot.png"
)

source_boxplot(
    episode_metrics, "reward_density_per_1000_raw", "Reward per 1,000 raw frames",
    "Breakout reward density by source",
    "03_reward_density_boxplot.png"
)

source_boxplot(
    episode_metrics, "first_reward_latency_raw", "First reward latency in raw frames",
    "Breakout first reward latency by source",
    "04_first_reward_latency_boxplot.png"
)

source_boxplot(
    episode_metrics, "life_losses", "Life losses",
    "Breakout life losses by source",
    "05_life_losses_boxplot.png"
)

In [ ]:
# Truncation-rate figure.
truncation_rate = (
    episode_metrics.assign(truncated=lambda d: d["end_type"].eq("truncated_at_cap"))
    .groupby("source")["truncated"]
    .mean()
    .reindex(SOURCE_ORDER)
    .reset_index()
)
truncation_rate.columns = ["source", "truncation_rate"]
truncation_rate.to_csv(TABLE_DIR / "21_truncation_rate.csv", index=False)

source_bar(
    truncation_rate, "source", "truncation_rate",
    "Breakout share of episodes reaching the evaluation cap",
    "Truncation rate",
    "06_truncation_rate_bar.png"
)

In [ ]:
# Return vs episode length.
fig, ax = plt.subplots(figsize=(7, 5))
for source in SOURCE_ORDER:
    sub = episode_metrics[episode_metrics["source"] == source]
    ax.scatter(sub["raw_frames"], sub["final_return"], label=source, alpha=0.8)
ax.set_xlabel("Episode length in raw ALE frames")
ax.set_ylabel("Episode return")
ax.set_title("Breakout return vs episode length")
ax.legend()
ax.grid(alpha=0.25)
save_show(fig, "07_return_vs_length_scatter.png")

In [ ]:
# Action distribution: time-weighted grouped bar chart.
# FIRE is excluded and proportions are renormalized over NOOP/RIGHT/LEFT.
piv = (
    action_distribution_time
    .pivot(index="action_label", columns="source", values="proportion")
    .reindex(list(REPORT_ACTION_LABELS.values()))
    .reindex(columns=SOURCE_ORDER)
)

fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(piv.index))
width = 0.25
for i, source in enumerate(SOURCE_ORDER):
    ax.bar(x + (i - 1) * width, piv[source].values, width, label=source)
ax.set_xticks(x)
ax.set_xticklabels(piv.index)
ax.set_ylabel("Proportion of non-FIRE aligned decision windows")
ax.set_title("Breakout movement/action distribution, time-weighted (FIRE excluded)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_show(fig, "08_action_distribution_time_weighted_no_fire.png")

In [ ]:
# Episode-averaged action distribution.
# FIRE is excluded and proportions are renormalized over NOOP/RIGHT/LEFT.
piv_ep = action_distribution_episode_avg.set_index("source").reindex(SOURCE_ORDER)[list(REPORT_ACTION_LABELS.values())].T

fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(piv_ep.index))
width = 0.25
for i, source in enumerate(SOURCE_ORDER):
    ax.bar(x + (i - 1) * width, piv_ep[source].values, width, label=source)
ax.set_xticks(x)
ax.set_xticklabels(piv_ep.index)
ax.set_ylabel("Mean episode-level proportion among non-FIRE windows")
ax.set_title("Breakout movement/action distribution, episode-averaged (FIRE excluded)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_show(fig, "09_action_distribution_episode_averaged_no_fire.png")

In [ ]:
# Action metric figures.
# These metrics are computed after excluding FIRE, so they describe paddle-control dynamics only.
source_boxplot(
    action_episode_metrics, "action_entropy_bits", "Action entropy (bits, FIRE excluded)",
    "Breakout action entropy by source (FIRE excluded)",
    "10_action_entropy_boxplot_no_fire.png"
)

source_boxplot(
    action_episode_metrics, "switching_rate", "Switching rate (FIRE excluded)",
    "Breakout switching rate by source (FIRE excluded)",
    "11_switching_rate_boxplot_no_fire.png"
)

source_boxplot(
    action_episode_metrics, "mean_run_length", "Mean run length (FIRE excluded)",
    "Breakout mean action run length by source (FIRE excluded)",
    "12_mean_run_length_boxplot_no_fire.png"
)

source_boxplot(
    action_episode_metrics, "trigram_diversity", "Trigram diversity (FIRE excluded)",
    "Breakout trigram diversity by source (FIRE excluded)",
    "13_trigram_diversity_boxplot_no_fire.png"
)

source_boxplot(
    action_episode_metrics, "fourgram_diversity", "Four-gram diversity (FIRE excluded)",
    "Breakout four-gram diversity by source (FIRE excluded)",
    "14_fourgram_diversity_boxplot_no_fire.png"
)


In [ ]:
# Plot top motifs for each source and n.
# FIRE is excluded before motif extraction, so motifs focus on NOOP/RIGHT/LEFT control patterns.
for n in [3, 4]:
    for source in SOURCE_ORDER:
        sub = motif_table[(motif_table["source"] == source) & (motif_table["n"] == n)].head(10).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, 4.8))
        ax.barh(sub["motif"], sub["proportion"])
        ax.set_xlabel("Proportion of all non-FIRE motifs")
        ax.set_title(f"Breakout top {n}-gram motifs, FIRE excluded: {source}")
        ax.grid(axis="x", alpha=0.25)
        save_show(fig, f"15_top_{n}gram_motifs_no_fire_{source}.png")

In [ ]:
# RAM PCA scatter sample.
fig, ax = plt.subplots(figsize=(7, 5.5))
for source in SOURCE_ORDER:
    sub = pca_sample[pca_sample["source"] == source]
    ax.scatter(sub["pc1"], sub["pc2"], s=5, alpha=0.35, label=source)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Breakout RAM-state PCA sample")
ax.legend(markerscale=3)
ax.grid(alpha=0.2)
save_show(fig, "16_ram_pca_sample.png")

In [ ]:
# Cluster visitation heatmaps.
# Matplotlib's default imshow colormap is used.
for table, name, title in [
    (cluster_shares_time_15, "time_weighted", "Breakout RAM cluster visitation, time-weighted"),
    (cluster_shares_episode_avg_15, "episode_averaged", "Breakout RAM cluster visitation, episode-averaged"),
]:
    fig, ax = plt.subplots(figsize=(9, 3.2))
    im = ax.imshow(table.values, aspect="auto")
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels(table.index)
    ax.set_xticks(np.arange(len(table.columns)))
    ax.set_xticklabels([str(c) for c in table.columns], rotation=0)
    ax.set_xlabel("Cluster")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Share")
    save_show(fig, f"17_cluster_heatmap_k15_{name}.png")

In [ ]:
# Cluster difference plots against Human.
for other in ["PPO", "DQN"]:
    diff = cluster_shares_time_15.loc[other] - cluster_shares_time_15.loc["Human"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(diff.index.astype(str), diff.values)
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Cluster")
    ax.set_ylabel(f"Share difference: {other} - Human")
    ax.set_title(f"Breakout RAM cluster over/under-visitation: {other} vs Human")
    ax.grid(axis="y", alpha=0.25)
    save_show(fig, f"18_cluster_difference_{other}_minus_Human.png")

## 10. Compact thesis-ready tables

These tables are useful for directly writing the Results section.

In [ ]:
# Compact performance table.
compact_performance = (
    episode_metrics.groupby("source")
    .agg(
        n_episodes=("episode", "nunique"),
        mean_return=("final_return", "mean"),
        sd_return=("final_return", "std"),
        median_return=("final_return", "median"),
        mean_raw_frames=("raw_frames", "mean"),
        sd_raw_frames=("raw_frames", "std"),
        mean_reward_density=("reward_density_per_1000_raw", "mean"),
        mean_first_reward_latency=("first_reward_latency_raw", "mean"),
        mean_life_losses=("life_losses", "mean"),
    )
    .reindex(SOURCE_ORDER)
    .round(3)
)
compact_performance.to_csv(TABLE_DIR / "22_compact_performance_table.csv")
compact_performance

In [ ]:
# Compact action-structure table.
# All metrics are computed after FIRE exclusion and therefore describe NOOP/RIGHT/LEFT paddle-control dynamics.
compact_action = (
    action_episode_metrics.groupby("source")
    .agg(
        action_entropy=("action_entropy_bits", "mean"),
        switching_rate=("switching_rate", "mean"),
        mean_run_length=("mean_run_length", "mean"),
        trigram_diversity=("trigram_diversity", "mean"),
        fourgram_diversity=("fourgram_diversity", "mean"),
    )
    .reindex(SOURCE_ORDER)
    .round(3)
)
compact_action.to_csv(TABLE_DIR / "23_compact_action_structure_table_no_fire.csv")
compact_action


In [ ]:
# Compact state-visitation table.
compact_state = ram_exact_overlap.copy()
compact_state.to_csv(TABLE_DIR / "24_compact_exact_ram_overlap_table.csv", index=False)
compact_state

In [ ]:
# Main interpretation-support table: combines performance, action, and state distances.
interpretation_rows = []

for source in SOURCE_ORDER:
    interpretation_rows.append({
        "source": source,
        "mean_return": compact_performance.loc[source, "mean_return"],
        "mean_reward_density": compact_performance.loc[source, "mean_reward_density"],
        "truncation_rate": float(truncation_rate.set_index("source").loc[source, "truncation_rate"]),
        "action_entropy_no_fire": compact_action.loc[source, "action_entropy"],
        "switching_rate_no_fire": compact_action.loc[source, "switching_rate"],
        "mean_run_length_no_fire": compact_action.loc[source, "mean_run_length"],
        "unique_exact_ram_states": int(ram_exact_overlap.loc[ram_exact_overlap["source_or_pair"] == source, "unique_exact_ram_states"].iloc[0]),
    })

interpretation_table = pd.DataFrame(interpretation_rows)
interpretation_table.to_csv(TABLE_DIR / "25_main_interpretation_table_no_fire_action_metrics.csv", index=False)
interpretation_table
